# RealSaS — Knight V9 Static Mechanics Microstep — Run All

**Notebook-first audit court. CPU-only. Drive-persistent. Resume-safe.**

Operator chain: `repartition + flip → collapse → vertex_cavity → edge_cavity`.

Run All is the intended execution mode. The notebook mounts Drive, checks out a pinned orchestration commit, runs a fail-closed preflight, resumes only hash-verified checkpoints, streams operator telemetry, and publishes every completed state back to Drive. It does not train models or mint product authority.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import shutil, subprocess, sys

BOOTSTRAP_COMMIT = '0579bf26af59496e1db6158b140ac05c8e8d8525'
BOOTSTRAP = Path('/content/realsas_v9_static_mechanics_bootstrap')
if BOOTSTRAP.exists():
    shutil.rmtree(BOOTSTRAP)
subprocess.run(['git','clone','--quiet','https://github.com/merynz/RealSaS-OPT.git',str(BOOTSTRAP)],check=True)
subprocess.run(['git','-C',str(BOOTSTRAP),'checkout','--quiet',BOOTSTRAP_COMMIT],check=True)
got=subprocess.check_output(['git','-C',str(BOOTSTRAP),'rev-parse','HEAD'],text=True).strip()
assert got==BOOTSTRAP_COMMIT,(got,BOOTSTRAP_COMMIT)
RUNNER=BOOTSTRAP/'tools/notebooks/run_knight_v9_static_mechanics_notebook_v1.py'
assert RUNNER.is_file(),RUNNER
subprocess.run([sys.executable,'-m','py_compile',str(RUNNER)],check=True)
print('NOTEBOOK_BOOTSTRAP_PREFLIGHT_PASS', got)


In [ ]:
DRIVE_BASE = '/content/drive/MyDrive/RealSaS_SUBJECT2_KNIGHT_DEMO_V2_20260924/KNIGHT_V9_STATIC_MECHANICS_NOTEBOOK_20261004'
RESET_CHECKPOINTS = False

cmd=[sys.executable,'-u',str(RUNNER),'--drive-base',DRIVE_BASE]
if RESET_CHECKPOINTS:
    cmd.append('--reset-checkpoints')
print('RUN_ALL_COMMAND', ' '.join(cmd), flush=True)
subprocess.run(cmd,cwd=BOOTSTRAP,check=True)


In [ ]:
import json
from pathlib import Path

latest_path=Path(DRIVE_BASE)/'RESULTS/LATEST_RUN.json'
assert latest_path.is_file(),latest_path
latest=json.loads(latest_path.read_text(encoding='utf-8'))
summary_path=Path(latest['result_dir'])/'RUN_ALL_SUMMARY.json'
summary=json.loads(summary_path.read_text(encoding='utf-8'))
print('\n=== VERIFIED DRIVE RESULT ===')
print('run_stamp:', summary['run_stamp'])
print('source_commit:', summary['source_commit'])
print('final_candidate_sha256:', summary['final_candidate_sha256'])
print('final_partition_sha256:', summary['final_partition_sha256'])
try:
    import pandas as pd
    display(pd.DataFrame(summary['steps']))
except Exception:
    print(json.dumps(summary['steps'],indent=2,sort_keys=True))
print('KNIGHT_V9_STATIC_MECHANICS_NOTEBOOK_VERIFIED_PASS')
